# Motex V4（已验证改进的默认组合）

V4 = V3（内置因果掩码 / 权重绑定 / 统一接口）+ 通过 A/B 实验验证的改进：

1. **MLA（Multi-head Latent Attention）**：KV 缓存降到 GQA 的 **1/8**（每 token 每层 1024B→128B）；域内验证略优、外推更优。
2. **QK-Norm（默认开）**：softmax 前对 Q/K 做 RMSNorm；域内验证最优 + 外推 CE **11.6→6.1**，零成本。
3. **bf16/AMP 训练（推荐模式）**：吞吐 **2×**、0 NaN、质量持平。

其余保持可选（负结果/场景性）：`rope_scaling`（RoPE 外推，本规模实验无收益）、`use_sdp`（真 FlashAttention，大模型/长上下文再开）。

> 本 notebook 只展示**模型本体**的构建与使用；训练/数据管线在仓库 `dev/`（不入库）。

In [ ]:
# 仓库根目录加入 sys.path（本仓库自包含）
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import torch

from motex_utils.motex_v4 import MotexV4

In [ ]:
# v4 默认：MLA + QK-Norm（attn='mla', qk_norm=True）
net = MotexV4(vocab_size=4096, d_model=256, num_layers=4, num_heads=4,
              num_kv_heads=2, ffn_hidden=512, dropout=0.1, max_seq_len=128)
print('参数(M):', round(sum(p.numel() for p in net.parameters()) / 1e6, 2))
print('注意层类型:', type(net.decoder.blks[0].attn).__name__, '| qk_norm =', net.decoder.blks[0].attn.qk_norm)
print('KV 缓存/层/token(B):', net.decoder.blks[0].attn.kv_cache_bytes_per_token())

x = torch.randint(5, 4096, (2, 128))
logits, state, aux = net(x, None, None)
print('train 前向 logits', tuple(logits.shape), ' 返回 3 元组')

## bf16 训练（推荐）

```python
with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
    logits, _, _ = net(x, None, None)
    l = loss(logits.reshape(-1, vocab), y.reshape(-1))
l.float().backward()   # bf16 不需要 GradScaler
```

In [ ]:
# 生成接口（与 v3 相同，KV-Cache 增量解码）
from motex_utils.motex_v3 import generate

class MiniTok:
    def __init__(self, chars):
        self.stoi = {s: i for i, s in enumerate(['<pad>', '<unk>', '<bos>', '<eos>', '\n'] + chars)}
        self.itos = {v: k for k, v in self.stoi.items()}
    def encode(self, t): return [self.stoi.get(c, 1) for c in t]

tok = MiniTok(list('你我他她说目的地方向'))
net2 = MotexV4(vocab_size=len(tok.stoi), d_model=128, num_layers=2, num_heads=4,
               num_kv_heads=1, ffn_hidden=256, dropout=0.0, max_seq_len=64).eval()
print(generate(net2, tok, '你', 30, 'cpu', temperature=0.9, top_k=10, repetition_penalty=1.1)[0][:40])